# Part 04 — Gradients & Backpropagation

Gradients are the **feedback signal** that tells a model which direction to adjust its parameters to reduce its mistakes.

```
Training loop (simplified):
  1. Forward pass  → compute output from current weights
  2. Loss          → measure how wrong the output is
  3. Backward pass → compute ∂loss/∂weight for every parameter  ← gradients
  4. Update        → weight -= learning_rate × gradient
  5. Repeat
```

### What this notebook covers
| Section | Key idea |
|---------|----------|
| Forward pass + loss | Compute output, measure error |
| Chain rule & backprop | How gradients flow backward through layers |
| Gradient descent | Using gradients to update weights |
| Training loop | Full convergence walkthrough with visualization |
| Learning rate | Too small, just right, too large — visual comparison |
| `torch.no_grad()` | Why and when to disable gradient tracking |
| Parameters & bias | The two learnable quantities in every layer |

---
## What is a Gradient?

A gradient is the **slope of the loss function** with respect to a parameter.

> Analogy: You're blindfolded on a hilly surface, trying to walk downhill. The gradient tells you: *"The ground slopes up to your left"* — so you step right. Repeat until you reach the valley (minimum loss).

### The chain rule in one line

For a neural network `loss = f(output) = f(g(w))`:

```
∂loss     ∂loss    ∂output
────── =  ────── × ───────    ← chain rule
 ∂w       ∂output    ∂w
```

**Concrete example** — model: `output = w × x + b`,  loss: `(output − target)²`

```
∂loss/∂output = 2 × (output − target)          ← derivative of squared error
∂output/∂w   = x                                ← derivative of linear layer

∂loss/∂w      = 2 × (output − target) × x      ← chain rule product

With: output=7, target=10, x=3:
  ∂loss/∂w  = 2 × (7 − 10) × 3 = 2 × (−3) × 3 = −18
  ∂loss/∂b  = 2 × (7 − 10) × 1 = −6
```

**What the sign tells you:**
- Gradient = −18 → loss *decreases* as w *increases* → increase w to reduce loss
- `new_w = old_w − lr × gradient = 2.0 − 0.1 × (−18) = 3.8`  ✓ moved toward 10

---
## Forward Pass + Backward Pass — PyTorch traces the math automatically

In [ ]:
import torch

# Initialize
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)
x = torch.tensor([3.0])
target = torch.tensor([10.0])
learning_rate = 0.01 # The learning rate needs to be small enough to avoid oscillation.

print("Training Progress:")
print("Iteration | Weight | Bias  | Output | Loss")
print("-" * 45)

for iteration in range(50):
    # Forward pass
    output = w * x + b
    loss = (output - target)**2
    
    print(f"{iteration:9d} | {w.item():6.2f} | {b.item():5.2f} | {output.item():6.2f} | {loss.item():6.2f}")
    
    # Backward pass
    if w.grad is not None:
        w.grad.zero_()
    if b.grad is not None:
        b.grad.zero_()
    
    loss.backward()
    
    # Update parameters
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Stop if close enough
    if abs(output.item() - target.item()) < 0.01:
        print(f"Converged at iteration {iteration}!")
        break

print(f"\nFinal: w={w.item():.3f}, b={b.item():.3f}")
print(f"Final output: {(w*x + b).item():.3f} (target: {target.item()})")

---
## Training Loop — Watching the Model Converge

---
## Visualizing Gradient Descent

The loss landscape for `loss = (w×3 + b − 10)²` is a bowl.  
Gradient descent rolls the parameter down the slope toward the minimum.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# ── Reproduce the training and record history ─────────────────────────────
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)
x = torch.tensor([3.0])
target = torch.tensor([10.0])
lr = 0.01

history = {"iter": [], "w": [], "b": [], "loss": []}

for i in range(50):
    output = w * x + b
    loss   = (output - target) ** 2

    history["iter"].append(i)
    history["w"].append(w.item())
    history["loss"].append(loss.item())

    if w.grad is not None: w.grad.zero_()
    if b.grad is not None: b.grad.zero_()
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    if abs(output.item() - target.item()) < 0.01:
        break

# ── Panel 1: Loss landscape (fixing b, varying w) ─────────────────────────
w_range = np.linspace(0.5, 5.5, 300)
b_fixed = 1.3   # approx final b
loss_landscape = [(wi * 3 + b_fixed - 10)**2 for wi in w_range]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Gradient Descent — Visual Walkthrough", fontsize=13, fontweight='bold')

# Loss landscape
ax1 = axes[0]
ax1.plot(w_range, loss_landscape, 'b-', linewidth=2, label='loss = (3w + b − 10)²')
ax1.scatter(history["w"], [(wi * 3 + b_fixed - 10)**2 for wi in history["w"]],
            c=range(len(history["w"])), cmap='Reds', s=40, zorder=5)
ax1.scatter(history["w"][0], (history["w"][0] * 3 + b_fixed - 10)**2,
            color='red', s=120, zorder=6, label=f'Start w={history["w"][0]:.1f}')
ax1.scatter(history["w"][-1], (history["w"][-1] * 3 + b_fixed - 10)**2,
            color='green', s=120, marker='*', zorder=6, label=f'End w={history["w"][-1]:.2f}')
ax1.set_xlabel("Weight (w)"); ax1.set_ylabel("Loss"); ax1.set_title("Loss Landscape\n(b fixed ≈ final value)")
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# Loss curve over iterations
ax2 = axes[1]
ax2.plot(history["iter"], history["loss"], 'b-o', markersize=3, linewidth=1.5)
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Loss")
ax2.set_title("Loss Curve\n(converges to 0)")
ax2.fill_between(history["iter"], history["loss"], alpha=0.15)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# Weight trajectory
ax3 = axes[2]
ax3.plot(history["iter"], history["w"], 'g-o', markersize=3, linewidth=1.5, label='weight')
ax3.axhline(3.0, color='gray', linestyle='--', alpha=0.7, label='target w≈3')
ax3.set_xlabel("Iteration"); ax3.set_ylabel("Weight value")
ax3.set_title("Weight Trajectory\n(approaches optimal ≈ 3)")
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("images/gradient_descent_viz.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"Converged in {len(history['iter'])} iterations")

---
## Learning Rate — The Most Important Hyperparameter

`lr` controls how big each update step is.  
Too small → painfully slow. Too large → overshoots and diverges.

In [ ]:
import torch, matplotlib.pyplot as plt

def run_training(lr, n_iter=60):
    """Train y = w*3 + b = 10 and return loss history."""
    w = torch.tensor([2.0], requires_grad=True)
    b = torch.tensor([1.0], requires_grad=True)
    x, target = torch.tensor([3.0]), torch.tensor([10.0])
    losses = []
    for _ in range(n_iter):
        output = w * x + b
        loss   = (output - target) ** 2
        losses.append(loss.item())
        if w.grad is not None: w.grad.zero_()
        if b.grad is not None: b.grad.zero_()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        if loss.item() > 1e6:   # diverging
            break
    return losses

configs = [
    (0.001, "#3498db", "lr=0.001  (too slow)"),
    (0.01,  "#2ecc71", "lr=0.01   (good)"),
    (0.05,  "#f39c12", "lr=0.05   (slightly fast)"),
    (0.12,  "#e74c3c", "lr=0.12   (diverges)"),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Learning Rate Comparison", fontsize=13, fontweight='bold')

ax1, ax2 = axes

for lr, color, label in configs:
    losses = run_training(lr)
    # Clip for display
    display = [min(l, 500) for l in losses]
    ax1.plot(display, color=color, linewidth=2, label=label)
    ax2.semilogy(display, color=color, linewidth=2, label=label)

for ax in [ax1, ax2]:
    ax.set_xlabel("Iteration"); ax.set_ylabel("Loss")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax1.set_title("Linear scale — divergence visible")
ax2.set_title("Log scale — convergence speed visible")

plt.tight_layout()
plt.savefig("images/learning_rate_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

print("Rule of thumb:")
print("  Start at lr=1e-3, try 1e-4 and 1e-2 and pick the fastest stable one.")
print("  Red flag: loss goes up instead of down → halve the learning rate.")

---
## `torch.no_grad()` — Training vs Inference

During **training** PyTorch builds a computation graph of every operation so it can  
backpropagate. During **inference** that graph is wasted memory and CPU time.

| | `requires_grad=True` (default) | `torch.no_grad()` |
|--|-------------------------------|-------------------|
| Purpose | Training | Inference / evaluation |
| Computation graph | Built and stored | Not built |
| Memory | Higher (all intermediates kept) | Lower |
| Speed | Slower | ~30% faster |
| `loss.backward()` | Works | Raises `RuntimeError` |

**Rule:** wrap every evaluation loop in `with torch.no_grad():`

---
## Memory: Computation Graph vs No Graph

Every intermediate tensor in the forward pass must be kept in memory for backprop.  
For a 12-layer BERT, that means retaining activations for every layer.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Memory Layout: With Gradients (Training) vs Without (Inference)",
             fontsize=12, fontweight='bold')

N_LAYERS = 12
layer_names = ["Embed"] + [f"L{i+1}" for i in range(N_LAYERS)] + ["Output"]

# ── Training: every layer activation stored ──────────────────────────────
ax1 = axes[0]
colors_train = ["#e74c3c"] * len(layer_names)   # red = stored
for i, (name, color) in enumerate(zip(layer_names, colors_train)):
    rect = plt.Rectangle((0, i * 0.7), 3, 0.55, color=color, alpha=0.8)
    ax1.add_patch(rect)
    ax1.text(1.5, i * 0.7 + 0.27, name, ha='center', va='center',
             color='white', fontsize=9, fontweight='bold')
    ax1.annotate("", xy=(3.3, i * 0.7 + 0.27), xytext=(3.05, i * 0.7 + 0.27),
                 arrowprops=dict(arrowstyle="->", color="gray", lw=1))
    ax1.text(3.4, i * 0.7 + 0.27, "grad_fn ✓", fontsize=7.5, color='#c0392b', va='center')

ax1.set_xlim(-0.2, 5.5); ax1.set_ylim(-0.4, len(layer_names) * 0.7 + 0.2)
ax1.set_title("WITH gradients (training)\nAll activations retained for backprop",
              fontsize=10, color='#c0392b')
ax1.axis('off')

# ── Inference: only output stored ────────────────────────────────────────
ax2 = axes[1]
for i, name in enumerate(layer_names):
    is_output = (i == len(layer_names) - 1)
    color = "#2ecc71" if is_output else "#95a5a6"
    alpha = 0.9 if is_output else 0.35
    rect = plt.Rectangle((0, i * 0.7), 3, 0.55, color=color, alpha=alpha)
    ax2.add_patch(rect)
    ax2.text(1.5, i * 0.7 + 0.27, name, ha='center', va='center',
             color='white' if is_output else '#555', fontsize=9,
             fontweight='bold' if is_output else 'normal')
    if is_output:
        ax2.text(3.4, i * 0.7 + 0.27, "kept ✓", fontsize=7.5, color='#27ae60', va='center')
    else:
        ax2.text(3.4, i * 0.7 + 0.27, "freed ✗", fontsize=7.5, color='#aaa', va='center')

ax2.set_xlim(-0.2, 5.5); ax2.set_ylim(-0.4, len(layer_names) * 0.7 + 0.2)
ax2.set_title("WITHOUT gradients (inference)\nOnly output kept — intermediates freed immediately",
              fontsize=10, color='#27ae60')
ax2.axis('off')

plt.tight_layout()
plt.savefig("images/gradient_memory.png", dpi=120, bbox_inches='tight')
plt.show()

# Memory estimate
seq_len, d_model, n_layers = 512, 768, 12
bytes_per_activation = seq_len * d_model * 4   # float32
train_mem_mb = bytes_per_activation * (n_layers + 2) / 1e6
infer_mem_mb = bytes_per_activation / 1e6
print(f"Activation memory (BERT, seq_len={seq_len}):")
print(f"  Training (all layers): ~{train_mem_mb:.1f} MB  (×{n_layers+2} layers)")
print(f"  Inference (output only): ~{infer_mem_mb:.1f} MB")
print(f"  Ratio: ~{train_mem_mb/infer_mem_mb:.0f}× more memory during training")

---
## Parameters & Bias — The Two Learnable Quantities

Every linear transformation in a neural network has exactly two types of learnable parameters:

```
output = input @ W + b

  W  (weight matrix) — scales and rotates the input
  b  (bias vector)   — shifts the output regardless of input
```

**Why bias matters:**
- Without bias: if `input = 0`, output is always `0` — the neuron can't fire independently
- With bias: the neuron has a "default activation level" it can adjust

```python
# In BERT's first attention layer (query projection):
W shape: [768, 768]  →  768 × 768 = 589,824 parameters
b shape: [768]       →  768 parameters
Total: 590,592 parameters in just this one sub-layer

# All parameters in a transformer block (BERT-base):
#   Attention Q/K/V + output:  4 × (768×768 + 768)    = 2,362,368
#   MLP FC1 + FC2:             (768×3072 + 3072) + (3072×768 + 768) = 4,722,432
#   LayerNorm ×2:              2 × (768 + 768)         = 3,072
#   ──────────────────────────────────────────────────────────────
#   Per block total:           ≈ 7,087,872
#   × 12 layers:               ≈ 85 million parameters
```

**During training**, gradients flow back to every `W` and every `b`.  
PyTorch's autograd engine computes `∂loss/∂W` and `∂loss/∂b` automatically via the chain rule.

---
## Summary

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| **Gradient** | `∂loss/∂param` — slope of loss w.r.t. parameter | Tells the optimizer which direction to move |
| **Backpropagation** | Chain rule applied layer by layer backward | Efficiently computes all gradients in one pass |
| **Gradient descent** | `param -= lr × gradient` | The update rule that makes models learn |
| **Learning rate** | Step size for each update | Too large → diverge; too small → slow |
| **Computation graph** | PyTorch's record of every operation | Required for backprop; skip it for inference |
| **`torch.no_grad()`** | Disables graph building | ~30% speedup + lower memory at inference time |
| **Weight (W)** | Matrix that transforms activations | Learned pattern detector |
| **Bias (b)** | Vector added to output | Shifts when a neuron fires |

```python
# The 4-line gradient descent loop — memorize this pattern:
output = model(x)               # 1. forward
loss   = criterion(output, y)   # 2. loss
optimizer.zero_grad()           # 3. clear old gradients
loss.backward()                 # 4. compute new gradients
optimizer.step()                # 5. update parameters
```